In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import plotly.express as px
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

In [22]:
df = pd.read_csv("hm_10k_working.csv")
print(df.head(5).to_string())

        t_dat                                                       customer_id  article_id     price  sales_channel_id   FN  Active club_member_status fashion_news_frequency   age                                                       postal_code  product_code               prod_name  product_type_no product_type_name  product_group_name  graphical_appearance_no graphical_appearance_name  colour_group_code colour_group_name  perceived_colour_value_id perceived_colour_value_name  perceived_colour_master_id perceived_colour_master_name  department_no          department_name index_code              index_name  index_group_no index_group_name  section_no                    section_name  garment_group_no garment_group_name                                                                                                                                                                                                                                   detail_desc
0  2019-09-13  215895f90002eb3d1a

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 35 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   t_dat                         10000 non-null  object 
 1   customer_id                   10000 non-null  object 
 2   article_id                    10000 non-null  int64  
 3   price                         10000 non-null  float64
 4   sales_channel_id              10000 non-null  int64  
 5   FN                            4243 non-null   float64
 6   Active                        4180 non-null   float64
 7   club_member_status            9975 non-null   object 
 8   fashion_news_frequency        9956 non-null   object 
 9   age                           9952 non-null   float64
 10  postal_code                   10000 non-null  object 
 11  product_code                  10000 non-null  int64  
 12  prod_name                     10000 non-null  object 
 13  pr

In [24]:
df.describe()

,article_id,price,sales_channel_id,FN,Active,age,product_code,product_type_no,graphical_appearance_no,colour_group_code,perceived_colour_value_id,perceived_colour_master_id,department_no,index_group_no,section_no,garment_group_no
count,1.000000e+04,10000.000000,10000.000000,4243.0,4180.0,9952.000000,10000.000000,10000.000000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,6.963121e+08,0.027955,1.704400,1.0,1.0,36.249799,696312.129900,246.575100,1.009711e+06,26.692100,3.269100,7.580400,2851.225600,2.447700,36.290300,1010.716100
std,1.334812e+08,0.019189,0.456335,0.0,0.0,12.945356,133481.220443,67.523398,1.749223e+04,26.604757,1.411979,5.045699,2089.861641,4.867939,23.170035,6.552801
min,1.087750e+08,0.000424,1.000000,1.0,1.0,17.000000,108775.000000,-1.000000,-1.000000e+00,-1.000000,-1.000000,-1.000000,1201.000000,1.000000,2.000000,1001.000000
25%,6.331448e+08,0.015758,1.000000,1.0,1.0,26.000000,633144.750000,253.000000,1.010012e+06,9.000000,2.000000,5.000000,1610.000000,1.000000,15.000000,1005.000000
50%,7.148260e+08,0.025407,2.000000,1.0,1.0,32.000000,714826.000000,265.000000,1.010016e+06,10.000000,4.000000,5.000000,1710.000000,1.000000,46.000000,1010.000000
75%,7.862080e+08,0.033881,2.000000,1.0,1.0,48.000000,786208.000000,273.000000,1.010016e+06,43.000000,4.000000,11.000000,3936.000000,2.000000,60.000000,1017.000000
max,9.419760e+08,0.337288,2.000000,1.0,1.0,83.000000,941976.000000,762.000000,1.010028e+06,93.000000,7.000000,20.000000,9989.000000,26.000000,97.000000,1025.000000


## Section 1: Null Handling
Review missing values, create simple flags where appropriate, and apply targeted imputations before moving to text standardization.

In [25]:
df = pd.read_csv('hm_10k_working.csv', dtype={'article_id': str, 'customer_id': str, 'product_code': str}, parse_dates=['t_dat'])
print("BEFORE NULL HANDLING:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print()

BEFORE NULL HANDLING:
FN                        5757
Active                    5820
club_member_status          25
fashion_news_frequency      44
age                         48
detail_desc                 41
dtype: int64



### FN Null Handling
`FN` has many missing values, which indicates that a customer is not enrolled in Fashion News. Create a binary subscriber flag from this column and treat nulls as not enrolled.

In [26]:
df['is_fn_subscriber'] = (df['FN'] == 1.0).astype(int)
df = df.drop('FN', axis=1)

In [27]:
df['is_active_customer'] = (df['Active'] == 1.0).astype(int)
df = df.drop('Active', axis=1)

In [28]:
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['price'] = pd.to_numeric(df['price'], errors='coerce')

scaler = StandardScaler()
scaled_age_price = scaler.fit_transform(df[['age', 'price']])

### KNN Imputation
Use nearest neighbors on the scaled `age` and `price` values to estimate missing ages.

In [29]:
imputer = KNNImputer(n_neighbors=5, weights='distance')
imputed_scaled = imputer.fit_transform(scaled_age_price)

### Map Back to `df`
Inverse-transform the scaled output and write only the imputed ages back into the missing rows of `df`.

In [30]:
imputed_age_price = scaler.inverse_transform(imputed_scaled)

missing_age_mask = df['age'].isna()
df.loc[missing_age_mask, 'age'] = imputed_age_price[missing_age_mask, 0]

print('Missing age values after KNN imputation:', df['age'].isna().sum())
print('Age filled directly in df using age and price.')

Missing age values after KNN imputation: 0
Age filled directly in df using age and price.


In [31]:
print('Age missing count:', df['age'].isna().sum())
print(df[['age', 'price']].head().to_string())

Age missing count: 0
    age     price
0  41.0  0.022017
1  30.0  0.028797
2  22.0  0.050831
3  21.0  0.027102
4  49.0  0.050831


### Remaining Null Fills
Handle the categorical and text columns with explicit fallback labels and combined descriptions.

In [32]:
df['fashion_news_frequency'] = df['fashion_news_frequency'].fillna('none')

In [33]:
df['club_member_status'] = df['club_member_status'].fillna('unknown')

In [34]:
df['detail_desc'] = df['detail_desc'].fillna(
    df['product_type_name'] + ' ' + df['garment_group_name']
)


In [35]:
df['has_description'] = 1

print("\nAFTER NULL HANDLING:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("✅ All critical nulls handled!")



AFTER NULL HANDLING:
Series([], dtype: int64)
✅ All critical nulls handled!


## Section 2: Standardize Text Columns
After null handling, normalize text fields to make categories consistent and replace `-1` placeholders in code-like columns with missing values.

### Text Standardization and `-1` Placeholder Fix
Lowercase and trim text columns, then convert `-1` values in numeric code fields to `NaN` so they are handled as true missing values.

In [ ]:
# Standardize all text columns (lowercase + strip whitespace)
text_cols = ['product_type_name', 'product_group_name', 'colour_group_name',
    'garment_group_name', 'department_name', 'section_name',
    'index_name', 'index_group_name', 'perceived_colour_value_name',
    'perceived_colour_master_name', 'graphical_appearance_name',
    'club_member_status', 'fashion_news_frequency']

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().str.strip()

# Replace -1 placeholders (unknown) with NaN in numeric code columns
placeholder_cols = ['product_type_no', 'graphical_appearance_no', 'colour_group_code',
    'perceived_colour_value_id', 'perceived_colour_master_id',
    'department_no', 'index_group_no', 'section_no', 'garment_group_no']

for col in placeholder_cols:
    if col in df.columns:
        count_before = (df[col] == -1).sum()
        df[col] = df[col].replace(-1, np.nan)
        if count_before > 0:
            print(f'{col}: replaced {count_before} -1s with NaN')

print('Text standardized + -1 placeholders fixed!')

product_type_no: replaced 33 -1s with NaN
graphical_appearance_no: replaced 3 -1s with NaN
colour_group_code: replaced 1 -1s with NaN
perceived_colour_value_id: replaced 1 -1s with NaN
perceived_colour_master_id: replaced 76 -1s with NaN
Text standardized + -1 placeholders fixed!


## Section 3: Price Log Transform and Drop Redundant Columns
Transform `price` with `log1p` to reduce skew, then remove duplicate code columns that already have readable name-based versions.

In [39]:
# Log transform price (right-skewed)
df['price_log'] = np.log1p(df['price'])

# Drop redundant numeric code columns (we have the name equivalents)
cols_to_drop = ['product_type_no', 'graphical_appearance_no', 'colour_group_code',
    'perceived_colour_value_id', 'perceived_colour_master_id',
    'department_no', 'index_group_no', 'section_no', 'garment_group_no',
    'index_code']

df = df.drop([col for col in cols_to_drop if col in df.columns], axis=1)

print(f'Dropped {len([c for c in cols_to_drop if c in df.columns])} redundant columns')
print(f'Final shape: {df.shape}')
print(f'Final columns ({df.shape[1]}):')
print(df.columns.tolist())

Dropped 0 redundant columns
Final shape: (10000, 27)
Final columns (27):
['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code', 'product_code', 'prod_name', 'product_type_name', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name', 'perceived_colour_master_name', 'department_name', 'index_name', 'index_group_name', 'section_name', 'garment_group_name', 'detail_desc', 'is_fn_subscriber', 'is_active_customer', 'has_description', 'price_log']


## Section 4: Save Cleaned File and Checkpoint
Save the cleaned dataset to a CSV file and print a final checkpoint so the preprocessing pipeline can be verified easily.

In [40]:
# Save cleaned dataframe
df.to_csv('hm_clean.csv', index=False)

print('Saved cleaned file to hm_clean.csv')
print('Final shape:', df.shape)
print('Checkpoint columns:', df.columns.tolist())

Saved cleaned file to hm_clean.csv
Final shape: (10000, 27)
Checkpoint columns: ['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code', 'product_code', 'prod_name', 'product_type_name', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name', 'perceived_colour_master_name', 'department_name', 'index_name', 'index_group_name', 'section_name', 'garment_group_name', 'detail_desc', 'is_fn_subscriber', 'is_active_customer', 'has_description', 'price_log']
